# Progress smoke test
Sleeps through fake stages (no model, CPU only) and posts each step to the run's ntfy.sh topic.

In [ ]:
%%writefile /tmp/progress.py
# say(stage, state, job=None, **extra): one line to this run's ntfy.sh topic. Never raises: progress must not break a run.
import json, sys, time, urllib.request
TOPIC = "__PROGRESS_TOPIC__"
def say(stage, state, job=None, **extra):
    if TOPIC.startswith("__"): return                      # not pushed through kaggle.sh: no relay
    ev = dict(t=round(time.time(), 1), stage=stage, state=state, job=job, **extra)
    try:
        req = urllib.request.Request(f"https://ntfy.sh/{TOPIC}", data=json.dumps(ev).encode(), method="POST")
        urllib.request.urlopen(req, timeout=10).read()
    except Exception as e:
        print("progress not sent:", e, file=sys.stderr)
if __name__ == "__main__":                                 # python /tmp/progress.py <stage> <state> [job]
    say(sys.argv[1], sys.argv[2], sys.argv[3] if len(sys.argv) > 3 else None)

In [ ]:
import sys; sys.path.insert(0, "/tmp")
from progress import say
say("setup", "start")

In [ ]:
import time
time.sleep(25); say("setup", "done")

In [ ]:
# fake stills: a model load, then one job each
say("stills", "start", note="loading model"); time.sleep(15); say("stills", "done", note="model loaded")
for sid, secs in [["p1", 20], ["p2", 20], ["p3", 20]]:
    say("stills", "start", job=sid); time.sleep(secs); say("stills", "done", job=sid, secs=secs)

In [ ]:
# fake clips, one of them fails like an out-of-memory clip
say("clips", "start", note="loading model"); time.sleep(15); say("clips", "done", note="model loaded")
for job in [["A-p1", 30], ["A-p2", 30, "out of GPU memory"], ["A-p3", 30]]:
    cid, secs = job[0], job[1]
    say("clips", "start", job=cid); time.sleep(secs)
    if len(job) > 2: say("clips", "error", job=cid, error=job[2])
    else: say("clips", "done", job=cid, secs=secs, vram_gb=13.6)

In [ ]:
say("pack", "start"); time.sleep(10)
open("/kaggle/working/progress-smoke-results.txt", "w").write("ok\n"); say("pack", "done")